# Trazador Cúbico   

## Dependencias y Librerias

In [74]:
%pip install numpy pandas matplotlib scipy openpyxl --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import urllib.request
from scipy.interpolate import CubicSpline, make_interp_spline
from openpyxl import load_workbook

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Lectura del archivo

In [75]:
archivo = "Hong Kong.xlsx"
url="https://raw.githubusercontent.com/Caro405/Pry_numerico/main/Hong%20Kong.xlsx"

# Descargar el archivo
urllib.request.urlretrieve(url, archivo)

# Comprobar que existe y tiene contenido
print("¿Existe el archivo?", os.path.exists(archivo))
print("Tamaño del archivo:", os.path.getsize(archivo), "bytes")

# Abrir el archivo de Excel y seleccionar la hoja activa
wb = load_workbook(archivo)
ws = wb.active

print("Archivo cargado correctamente desde GitHub.")
print("Hoja activa:", ws.title)
print("Número de filas:", ws.max_row)


¿Existe el archivo? True
Tamaño del archivo: 9918 bytes
Archivo cargado correctamente desde GitHub.
Hoja activa: Hoja1
Número de filas: 31


## Detección de los datos subrayados

In [76]:
anios_sub_auto = []
datos_anios = []
datos_poblacion = []

for fila in range(2, ws.max_row + 1):
    celda_anio = ws.cell(row=fila, column=2)
    celda_pob = ws.cell(row=fila, column=3)

    if celda_anio.value is not None and celda_pob.value is not None:
        anio = int(celda_anio.value)
        pob = float(celda_pob.value)

        datos_anios.append(anio)
        datos_poblacion.append(pob)

        # Revisar el color de fondo de la celda del año
        color_fondo = celda_anio.fill.start_color

        es_subrayado = False

        if color_fondo.type == 'rgb':
            # Filtrar los colores blancos (FFFFFFFF) y negros (00000000)
            if color_fondo.rgb not in ['FFFFFFFF', '00000000']:
                es_subrayado = True
        elif color_fondo.type == 'theme':
              if color_fondo.theme not in [0, 1]:
                es_subrayado = True

        if es_subrayado:
            anios_sub_auto.append(anio)

print(f"Años detectados automáticamente como subrayados: {anios_sub_auto}")

df = pd.DataFrame({"Año": datos_anios, "Población": datos_poblacion})

Años detectados automáticamente como subrayados: [1995, 2000, 2008, 2015, 2022]


## Separación de los datos subrayados

In [77]:
mask_sub = df["Año"].isin(anios_sub_auto)
df_train = df[~mask_sub].reset_index(drop=True)   # 25 datos a usar
df_test  = df[ mask_sub].reset_index(drop=True)   # 5 datos

x_train = df_train["Año"].values
y_train = df_train["Población"].values
x_test = df_test["Año"].values
y_exacto = df_test["Población"].values



print(f"Datos de prueba (subrayados): {len(df_test)}")
print(df_test)
print(f"\nDatos de entrenamiento (NO subrayados): {len(df_train)}")
print(df_train)

Datos de prueba (subrayados): 5
    Año  Población
0  1995  6177105.0
1  2000  6704665.0
2  2008  7020689.0
3  2015  7366411.0
4  2022  7465913.0

Datos de entrenamiento (NO subrayados): 25
     Año  Población
0   1994  6042259.0
1   1996  6302469.0
2   1997  6416674.0
3   1998  6521331.0
4   1999  6616524.0
5   2001  6774243.0
6   2002  6821412.0
7   2003  6858906.0
8   2004  6888164.0
9   2005  6908483.0
10  2006  6931089.0
11  2007  6971344.0
12  2009  7064464.0
13  2010  7101614.0
14  2011  7142969.0
15  2012  7201059.0
16  2013  7264330.0
17  2014  7319502.0
18  2016  7408006.0
19  2017  7444889.0
20  2018  7471190.0
21  2019  7485817.0
22  2020  7490240.0
23  2021  7484604.0
24  2023  7442738.0


## Trazador cúbico 

### Construcción del polinomio 

 En cada intervalo $[x_i,x_{i+1}]$, se construye un polinomio de la forma:

$
S_i(x)=a_i+b_i(x-x_i)+c_i(x-x_i)^2+d_i(x-x_i)^3
$

### Función CubicSpline de Python

La función CubicSpline de la biblioteca scipy.interpolate construye automáticamente el trazador cúbico a partir de los datos proporcionados. En este caso:

$ trazador = CubicSpline(x_train, y_train, bc_type='natural')
y_aprox_spline = trazador(x_test) $

 - x_train y y_train: contienen los años y las poblaciones utilizadas para construir el trazador.
 - bc_type='natural': establece condiciones de frontera naturales, haciendo que la segunda derivada sea cero en los extremos del intervalo.
 - trazador(x_test): evalúa el trazador en los años de prueba para estimar sus poblaciones.

In [78]:
# 1. Obtener los datos de entrenamiento
x_train = df_train["Año"].values
y_train = df_train["Población"].values

# 2. Obtener los años subrayados que se quieren aproximar
x_test = df_test["Año"].values

# 3. Crear el trazador cúbico natural
trazador = CubicSpline(
    x_train,
    y_train,
    bc_type="natural"
)

# 4. Calcular las aproximaciones
y_aprox_spline = trazador(x_test)

# 5. Organizar los resultados en una tabla
resultados = pd.DataFrame({
    "Año Subrayado": x_test.astype(int),
    "Aproximación (Spline)": y_aprox_spline.round(2) # Aproximación redondeada a 2 decimales
})

    # 6. Mostrar los resultados
print("=== RESULTADOS DEL TRAZADOR CÚBICO ===")
display(resultados)


=== RESULTADOS DEL TRAZADOR CÚBICO ===


,Año Subrayado,Aproximación (Spline)
0,1995,6176009.02
1,2000,6703882.61
2,2008,7019376.64
3,2015,7366447.07
4,2022,7467114.59


## Error Relativo 
### Fórmula 
$ |\frac {(Valor Exacto - Valor Aproximado Spline)} {Valor Exacto} | *100$

In [79]:
error_relativo_spline = np.abs((y_exacto - y_aprox_spline) / y_exacto) * 100

tabla_spline = pd.DataFrame({
    "Año": x_test.astype(int),
    "Población Exacta": y_exacto.astype(int),
    "Aproximación (Spline)": y_aprox_spline.round(2),
    "Error Relativo (%)": error_relativo_spline.round(6)
})

print("=== COMPARACIÓN: TRAZADOR CÚBICO ===")
display(tabla_spline)

print(f"Error relativo promedio: {error_relativo_spline.mean():.6f} %")


=== COMPARACIÓN: TRAZADOR CÚBICO ===


,Año,Población Exacta,Aproximación (Spline),Error Relativo (%)
0,1995,6177105,6176009.02,0.017743
1,2000,6704665,6703882.61,0.011669
2,2008,7020689,7019376.64,0.018693
3,2015,7366411,7366447.07,0.000490
4,2022,7465913,7467114.59,0.016094


Error relativo promedio: 0.012938 %
